In [1]:
import os
from pathlib import Path
import pandas as pd
import re

In [2]:
def split_img_nums(img_nums):
    if isinstance(img_nums, str):
        if ',' in img_nums:
            return list(map(str.strip, img_nums.split(",")))
        else:
            return [img_nums.strip()]
    elif isinstance(img_nums, int):
        return [str(img_nums)]

In [3]:
project_dir = Path("./project_dir")

In [5]:
df = pd.read_excel(project_dir / Path("H-48152_Lesion_Data") / Path("Lesion-Image Data.xlsx"))
df = df.rename(columns={c : c.replace(" ", "_") for c in df.columns})
df['Lesion_HRME_Image'] = df['Lesion_HRME_Image'].apply(split_img_nums)
df = df.explode('Lesion_HRME_Image').reset_index(drop=True)
df = df.sort_values(by=['Subject_ID', 'Lesion_HRME_Image'], ascending=[True, True])
df["Lesion_HRME_Image"].isnull().any()

np.False_

In [7]:
lesion_data_dir = project_dir / Path("H-48152_Lesion_Data")

In [8]:
def find_image(pt_folderpath, img_num, img_files=None):
    if img_files is None:
        img_files = []
    files = os.listdir(pt_folderpath)
    for file in files:
        filepath = pt_folderpath / Path(file)
        if filepath.is_dir() and "Images" in file:
            img_files = find_image(filepath, img_num, img_files=img_files)
        elif str(filepath).endswith(".png") and f"image{img_num}_" in str(filepath):
            img_files.append(filepath)
        else:
            continue
    return img_files


In [9]:
for i, row in df.iterrows():
    ptid, img_num = row["Subject_ID"], row["Lesion_HRME_Image"]
    image_filepath = find_image(lesion_data_dir / Path(ptid), img_num)
    if len(image_filepath)==1:
        df.loc[i, "Lesion_HRME_Image_Filepath"] = str(image_filepath[0])
    else:
        print(lesion_data_dir / Path(ptid), img_num)

df = df[~df["Lesion_HRME_Image_Filepath"].isnull()].reset_index(drop=True)

project_dir/H-48152_Lesion_Data/B010 3
project_dir/H-48152_Lesion_Data/B010 4
project_dir/H-48152_Lesion_Data/B010 5
project_dir/H-48152_Lesion_Data/B010 6


In [10]:
masks_dir = project_dir / Path('H-48152_Lesion_Data_Masks')
for i, row in df.iterrows():
    prefix, suffix = re.findall("(image\d+)(.*)", row["Lesion_HRME_Image_Filepath"])[0]
    subject_id = row["Subject_ID"]
    mask_fname = subject_id + "_" + prefix + "_Mask" + suffix
    df.loc[i, "Lesion_HRME_Mask_Filepath"] = str(masks_dir / Path(mask_fname))
df["Lesion_HRME_Mask_Filepath"].isnull().any()

np.False_

In [11]:
df

,Subject_ID,Lesion_#,Lesion_Level,Lesion_Quadrant,Lesion_Container,Lesion_Read,Lesion_Plan,Lesion_HRME_Image,Favorite_Lesion_HRME_Image_#,Local_Pathology,Local_Pathology_Final,Lesion_HRME_Image_Filepath,Lesion_HRME_Mask_Filepath
0,B005,1,39,6.0,B,LGD,Biopsy,1,2,Gastric type mucosa with foveolar hyperplasia ...,0,project_dir/H-48152_Lesion_Data/B005/B005 Imag...,project_dir/H-48152_Lesion_Data_Masks/B005_ima...
1,B005,1,39,6.0,B,LGD,Biopsy,2,2,Gastric type mucosa with foveolar hyperplasia ...,0,project_dir/H-48152_Lesion_Data/B005/B005 Imag...,project_dir/H-48152_Lesion_Data_Masks/B005_ima...
2,B005,2,35,6.0,C,Non-neoplastic,Biopsy,3,3,"Esophagus, at 35 cm, biopsy: - Gastric type ...",0,project_dir/H-48152_Lesion_Data/B005/B005 Imag...,project_dir/H-48152_Lesion_Data_Masks/B005_ima...
3,B007,1,36,10.0,A,Non-neoplastic,ESD,2,4,Intramucosal poorly differentiated adenocarcin...,1,project_dir/H-48152_Lesion_Data/B007/Represent...,project_dir/H-48152_Lesion_Data_Masks/B007_ima...
4,B007,1,36,10.0,A,Non-neoplastic,ESD,3,4,Intramucosal poorly differentiated adenocarcin...,1,project_dir/H-48152_Lesion_Data/B007/Represent...,project_dir/H-48152_Lesion_Data_Masks/B007_ima...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,T008,3,34,12.0,A,Non-neoplastic,Biopsy,5,5,BARRETT'S ESOPHAGUS - NEGATIVE FO...,0,project_dir/H-48152_Lesion_Data/T008/Represent...,project_dir/H-48152_Lesion_Data_Masks/T008_ima...
99,T008,4,32,3.0,B,Non-neoplastic,Biopsy,6,6,- BARRETT'S ESOPHAGUS - NEGATIVE...,0,project_dir/H-48152_Lesion_Data/T008/Represent...,project_dir/H-48152_Lesion_Data_Masks/T008_ima...
100,T008,5,32,6.0,B,Non-neoplastic,Biopsy,7,8,- BARRETT'S ESOPHAGUS - NEGATIVE...,0,project_dir/H-48152_Lesion_Data/T008/Represent...,project_dir/H-48152_Lesion_Data_Masks/T008_ima...
101,T008,5,32,6.0,B,Non-neoplastic,Biopsy,8,8,- BARRETT'S ESOPHAGUS - NEGATIVE...,0,project_dir/H-48152_Lesion_Data/T008/Represent...,project_dir/H-48152_Lesion_Data_Masks/T008_ima...


In [12]:
df.to_csv("./HRME_image_annotations_dataset.csv", index=False)